# Phased Single-Agent Training

Train a single RL jockey against BT opponents using **phased reward shaping**.
Each reward phase adds signals on top of the previous one. Obs space and physics are unchanged across all phases.

**Reward phases:**
- **Phase 1** — Race & Kick: forward progress, speed, pacing (cruise/kick), placement, stamina budget, terminals
- **Phase 2** — +Cornering: adds inside-line bonus, outside penalty, steering, straight pre-positioning
- **Phase 3** — +Archetype/Skills: adds archetype shaping and skill-conditioned bonuses

**Curriculum stages** (per phase):
1. **Straight** (400m) — learn stamina pacing
2. **Oval** (905m) — learn cornering + inside lines
3. **All tracks** — generalize across layouts
4. **Large fields** (8 horses) — positioning and overtaking

**How to use:**
1. Set `REWARD_PHASE` and `STAGE` in the Configuration cell
2. Run all cells top to bottom
3. If runtime disconnects, reconnect and run all — it auto-resumes from Drive

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running
# ============================================================

# Reward phase: 1=race/kick only, 2=+cornering, 3=+archetype/skills
REWARD_PHASE = 1

# Training stage: 1=straight, 2=oval, 3=all tracks, 4=large fields
STAGE = 1

# Stage configs: (tracks, horses, timesteps)
STAGE_CONFIGS = {
    1: {
        "tracks": ["tracks/curriculum_1_straight.json"],
        "horses": 4,
        "timesteps": 500_000,
        "description": "Straight 400m — learn stamina pacing",
    },
    2: {
        "tracks": ["tracks/test_oval.json"],
        "horses": 4,
        "timesteps": 1_000_000,
        "description": "Test Oval 905m — learn cornering",
    },
    3: {
        "tracks": None,  # None = all tracks
        "horses": 4,
        "timesteps": 2_000_000,
        "description": "All tracks — generalize",
    },
    4: {
        "tracks": None,
        "horses": 8,
        "timesteps": 2_000_000,
        "description": "All tracks, 8 horses — positioning & overtaking",
    },
}

# BT opponents on/off
BT_OPPONENTS = True

# Parallel envs (Colab usually has 2 CPUs, use 2-4)
N_ENVS = 4

# Checkpoint dir on Google Drive (organized by phase)
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints"
DRIVE_DIR = f"{DRIVE_BASE}/phase_{REWARD_PHASE}"

# Resume: 'auto' finds latest .zip, 'prev_stage' loads previous stage's final model,
#         'prev_phase' loads previous phase's final stage model
RESUME = 'auto'

# Learning rate — lower for Phase 2+ fine-tuning
LR_BY_PHASE = {
    1: 3e-4,
    2: 1e-4,
    3: 5e-5,
}

# Logging
LOG_EVERY = 10          # print every N episodes
SAVE_EVERY = 50_000     # checkpoint every N timesteps

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q "stable-baselines3>=2.3.0" "gymnasium>=1.0.0" "torch>=2.0.0" "onnx>=1.21.0" "onnxruntime>=1.24.4" "onnxscript>=0.6.2"

# Remove old gym package (triggers deprecation warning)
!pip uninstall -q -y gym 2>/dev/null || true

# Install the project in editable mode
!pip install -q -e .

# Suppress warnings from transitive imports
import warnings
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

## Resolve Stage Config & Checkpoint

In [ ]:
import glob
from pathlib import Path

cfg = STAGE_CONFIGS[STAGE]
stage_dir = os.path.join(DRIVE_DIR, f"stage_{STAGE}")
os.makedirs(stage_dir, exist_ok=True)

lr = LR_BY_PHASE[REWARD_PHASE]

# Resolve tracks
if cfg["tracks"] is None:
    track_paths = sorted(glob.glob("tracks/*.json"))
else:
    track_paths = cfg["tracks"]

print(f"Reward Phase {REWARD_PHASE}: {'Race/Kick' if REWARD_PHASE == 1 else '+Cornering' if REWARD_PHASE == 2 else '+Archetype/Skills'}")
print(f"Stage {STAGE}: {cfg['description']}")
print(f"Tracks: {len(track_paths)} ({', '.join(Path(t).stem for t in track_paths)})")
print(f"Horses: {cfg['horses']}")
print(f"Timesteps: {cfg['timesteps']:,}")
print(f"Learning rate: {lr}")
print(f"BT opponents: {'on' if BT_OPPONENTS else 'off'}")
print(f"Checkpoint dir: {stage_dir}")
print()

# Find checkpoint to resume from
restore_path = None

if RESUME == 'auto':
    # Look for latest checkpoint in current stage dir
    zips = sorted(glob.glob(os.path.join(stage_dir, "*.zip")), key=os.path.getmtime)
    if zips:
        restore_path = zips[-1]
        print(f"Auto-resume: {restore_path}")
    elif STAGE > 1:
        # Try loading previous stage's final model (same phase)
        prev_final = os.path.join(DRIVE_DIR, f"stage_{STAGE-1}", "final_model.zip")
        if os.path.exists(prev_final):
            restore_path = prev_final
            print(f"Loading previous stage: {restore_path}")
        else:
            print("No checkpoint found — starting fresh")
    elif REWARD_PHASE > 1:
        # Phase 2+, Stage 1: try loading previous phase's last stage final model
        prev_phase_dir = f"{DRIVE_BASE}/phase_{REWARD_PHASE - 1}"
        # Find the highest stage with a final_model in the previous phase
        for s in [4, 3, 2, 1]:
            prev_final = os.path.join(prev_phase_dir, f"stage_{s}", "final_model.zip")
            if os.path.exists(prev_final):
                restore_path = prev_final
                print(f"Loading from Phase {REWARD_PHASE - 1} Stage {s}: {restore_path}")
                break
        if not restore_path:
            print(f"No Phase {REWARD_PHASE - 1} checkpoint found — starting fresh")
    else:
        print("No checkpoint found — starting fresh")
elif RESUME == 'prev_stage' and STAGE > 1:
    prev_final = os.path.join(DRIVE_DIR, f"stage_{STAGE-1}", "final_model.zip")
    if os.path.exists(prev_final):
        restore_path = prev_final
        print(f"Loading previous stage: {restore_path}")
    else:
        print(f"Previous stage model not found at {prev_final} — starting fresh")
elif RESUME == 'prev_phase' and REWARD_PHASE > 1:
    prev_phase_dir = f"{DRIVE_BASE}/phase_{REWARD_PHASE - 1}"
    for s in [4, 3, 2, 1]:
        prev_final = os.path.join(prev_phase_dir, f"stage_{s}", "final_model.zip")
        if os.path.exists(prev_final):
            restore_path = prev_final
            print(f"Loading from Phase {REWARD_PHASE - 1} Stage {s}: {restore_path}")
            break
    if not restore_path:
        print(f"No Phase {REWARD_PHASE - 1} checkpoint found — starting fresh")
elif RESUME and RESUME not in ('auto', 'prev_stage', 'prev_phase'):
    restore_path = RESUME
    print(f"Resuming from: {restore_path}")
else:
    print("Starting fresh")

## Training

In [ ]:
import time
import json
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv

from horse_racing.engine import EngineConfig
from horse_racing.env import HorseRacingSingleEnv


class ColabLoggingCallback(BaseCallback):
    """Logs episode stats and saves history to Drive."""

    def __init__(self, log_every=10, total_timesteps=1_000_000, history_path=None):
        super().__init__(verbose=0)
        self._log_every = log_every
        self._total_timesteps = total_timesteps
        self._history_path = history_path
        self._ep_count = 0
        self._start = time.time()
        self.history = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._ep_count += 1
                ep = info["episode"]
                self.history.append({
                    "episode": self._ep_count,
                    "timesteps": self.num_timesteps,
                    "reward": float(ep["r"]),
                    "length": int(ep["l"]),
                })
                if self._ep_count % self._log_every == 0:
                    elapsed = time.time() - self._start
                    ts = self.num_timesteps
                    rate = ts / elapsed if elapsed > 0 else 0
                    pct = ts / self._total_timesteps * 100
                    eta_s = (self._total_timesteps - ts) / rate if rate > 0 else 0
                    eta_m = eta_s / 60
                    # Running avg of last 10 episodes
                    recent = self.history[-10:]
                    avg_r = np.mean([h["reward"] for h in recent])
                    avg_l = np.mean([h["length"] for h in recent])
                    print(
                        f"  [{pct:5.1f}%] ep {self._ep_count:5d} | "
                        f"reward: {avg_r:8.1f} | "
                        f"len: {avg_l:6.0f} | "
                        f"ts: {ts:>10,} | "
                        f"{rate:,.0f} ts/s | "
                        f"ETA: {eta_m:.0f}m"
                    )
                # Save history periodically
                if self._history_path and self._ep_count % (self._log_every * 5) == 0:
                    with open(self._history_path, "w") as f:
                        json.dump(self.history, f)
        return True


# Create vectorized env — wrap each with Monitor for episode stats
engine_config = EngineConfig(horse_count=cfg["horses"])

def make_env(rank):
    def _init():
        track = track_paths[rank % len(track_paths)]
        env = HorseRacingSingleEnv(
            track_path=track,
            config=engine_config,
            bt_opponents=BT_OPPONENTS,
            reward_phase=REWARD_PHASE,
        )
        return Monitor(env)
    return _init

vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])

# PPO hyperparameters
policy_kwargs = dict(net_arch=[256, 256])

if restore_path:
    print(f"Loading model from {restore_path}")
    model = PPO.load(restore_path, env=vec_env, device="cpu")
    model.learning_rate = lr
else:
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=lr,
        gamma=0.995,
        gae_lambda=0.95,
        n_steps=2048,
        batch_size=512,
        n_epochs=10,
        clip_range=0.2,
        ent_coef=0.02,
        policy_kwargs=policy_kwargs,
        verbose=0,
        device="cpu",
    )

print(f"Model params: {sum(p.numel() for p in model.policy.parameters()):,}")
print()

In [ ]:
# --- Training loop ---
history_path = os.path.join(stage_dir, "history.json")

checkpoint_cb = CheckpointCallback(
    save_freq=max(SAVE_EVERY // N_ENVS, 1),
    save_path=stage_dir,
    name_prefix="checkpoint",
)
logging_cb = ColabLoggingCallback(
    log_every=LOG_EVERY,
    total_timesteps=cfg["timesteps"],
    history_path=history_path,
)

print(f"Training Phase {REWARD_PHASE} / Stage {STAGE}: {cfg['description']}")
print(f"Target: {cfg['timesteps']:,} timesteps | LR: {lr}")
print(f"Checkpoints -> {stage_dir}")
print()

start_time = time.time()

try:
    model.learn(
        total_timesteps=cfg["timesteps"],
        callback=[checkpoint_cb, logging_cb],
        reset_num_timesteps=False if restore_path else True,
    )
except KeyboardInterrupt:
    print("\nTraining interrupted.")

# Save final model
final_path = os.path.join(stage_dir, "final_model")
model.save(final_path)

# Save history
with open(history_path, "w") as f:
    json.dump(logging_cb.history, f)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"Phase {REWARD_PHASE} / Stage {STAGE} complete")
print(f"  Time: {elapsed/60:.1f} min")
print(f"  Episodes: {logging_cb._ep_count}")
print(f"  Final model: {final_path}.zip")
print(f"{'='*60}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

if logging_cb.history:
    rewards = [h["reward"] for h in logging_cb.history]
    lengths = [h["length"] for h in logging_cb.history]

    # Smooth with rolling window
    window = min(50, len(rewards) // 4) or 1
    smooth_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smooth_l = np.convolve(lengths, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(rewards, alpha=0.2, color='blue')
    ax1.plot(range(window-1, len(rewards)), smooth_r, color='blue', linewidth=2)
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.set_title(f'Phase {REWARD_PHASE} / Stage {STAGE} — Episode Reward')
    ax1.grid(True, alpha=0.3)

    ax2.plot(lengths, alpha=0.2, color='green')
    ax2.plot(range(window-1, len(lengths)), smooth_l, color='green', linewidth=2)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Length (steps)')
    ax2.set_title(f'Phase {REWARD_PHASE} / Stage {STAGE} — Episode Length')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(stage_dir, f"phase_{REWARD_PHASE}_stage_{STAGE}_curves.png"), dpi=100)
    plt.show()

    print(f"Last 20 episodes: reward={np.mean(rewards[-20:]):.1f}, length={np.mean(lengths[-20:]):.0f}")
else:
    print("No training history to plot.")

## Quick Evaluation

In [ ]:
# Evaluate the trained model on a test race
from horse_racing.engine import HorseRacingEngine
from horse_racing.types import HorseAction

eval_track = "tracks/tokyo.json"
eval_horses = cfg["horses"]
eval_episodes = 5

print(f"Evaluating on {eval_track} ({eval_episodes} episodes, {eval_horses} horses)")
print(f"Reward phase: {REWARD_PHASE}")
print()

results = []
for ep in range(eval_episodes):
    env = HorseRacingSingleEnv(
        track_path=eval_track,
        config=EngineConfig(horse_count=eval_horses),
        bt_opponents=True,
        reward_phase=REWARD_PHASE,
    )
    obs, _ = env.reset()
    total_reward = 0
    speeds = []

    for step in range(5000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

        o = env.engine.get_observations()[0]
        speeds.append(o["tangential_vel"])

        if terminated or truncated:
            break

    o = env.engine.get_observations()[0]
    placements = env.engine.get_placements()
    stam = o["stamina_ratio"]
    finished = o.get("finished", False)
    progress = o["track_progress"]

    results.append({
        "finished": finished,
        "progress": progress,
        "reward": total_reward,
        "avg_speed": np.mean(speeds),
        "final_stamina": stam,
        "placement": placements[0],
        "steps": step + 1,
    })

    status = "FINISHED" if finished else f"{progress:.0%}"
    print(
        f"  ep {ep+1}: {status} | "
        f"place={placements[0]}/{eval_horses} | "
        f"reward={total_reward:.0f} | "
        f"speed={np.mean(speeds):.1f} | "
        f"stamina={stam:.0%} | "
        f"steps={step+1}"
    )
    env.close()

print()
comp = sum(1 for r in results if r["finished"]) / len(results)
wins = sum(1 for r in results if r["placement"] == 1)
print(f"Completion: {comp:.0%}")
print(f"Wins: {wins}/{len(results)} ({wins/len(results):.0%})")
print(f"Mean reward: {np.mean([r['reward'] for r in results]):.1f}")
print(f"Mean speed: {np.mean([r['avg_speed'] for r in results]):.1f} m/s")
print(f"Mean final stamina: {np.mean([r['final_stamina'] for r in results]):.0%}")
print(f"Mean placement: {np.mean([r['placement'] for r in results]):.1f}/{eval_horses}")

## Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from horse_racing.types import OBS_SIZE


class SB3PolicyWrapper(nn.Module):
    """Wraps SB3 PPO actor for ONNX export: obs -> remapped action.

    The policy network outputs raw actions in [-1, 1] (tanh-ish).
    This wrapper bakes in the action remapping so the ONNX model
    directly outputs physics-ready values:
        tangential: [-1, 1] -> [-2, 7]  (center 2.5, half-range 4.5)
        normal:     [-1, 1] -> [-3, 3]  (center 0, half-range 3)
    """
    def __init__(self, policy):
        super().__init__()
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, obs):
        features = self.mlp_extractor.forward_actor(obs)
        raw = self.action_net(features)
        # Remap: same as HorseRacingSingleEnv.remap_action
        tang = raw[:, 0:1] * 4.5 + 2.5   # [-1,1] -> [-2, 7]
        norm = raw[:, 1:2] * 3.0          # [-1,1] -> [-3, 3]
        return torch.cat([tang, norm], dim=1)


# Move to CPU for ONNX export
wrapper = SB3PolicyWrapper(model.policy).cpu()
wrapper.eval()

dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)
with torch.no_grad():
    test_out = wrapper(dummy)
    print(f"Test output (remapped): {test_out.numpy()}")
    print(f"  tang={test_out[0,0].item():.2f} (range [-2, 7]), norm={test_out[0,1].item():.2f} (range [-3, 3])")

onnx_path = os.path.join(stage_dir, f"stage_{STAGE}_jockey.onnx")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"],
    output_names=["action"],
    dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)

# Verify
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
result = session.run(None, {"obs": np.zeros((1, OBS_SIZE), dtype=np.float32)})
print(f"ONNX output (remapped): {result[0]}")
print(f"\nExported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")

## Cleanup

In [ ]:
vec_env.close()
print("Done.")